In [2]:
# Initialisation de la session Spark
# Master : spark://spark-master:7077
# Systeme de fichiers : HDFS (namenode:9000)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HDFS_Partitioning_Benchmark") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print(spark)

In [4]:
# Lecture du dataset CSV depuis HDFS
# header : premiere ligne = noms des colonnes
# inferSchema : Spark devine automatiquement les types
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("hdfs://namenode:9000/2023_Yellow_Taxi_Trip_Data.csv")

print("Lecture reussie")

Lecture reussie


In [6]:
# Comptage total des lignes
# count() declenche la lecture reelle du fichier depuis HDFS
print(f"Nombre de lignes: {df.count()}")

Nombre de lignes: 4843349


In [8]:
# Affichage des 5 premieres lignes
# truncate=False : affiche le contenu complet sans couper les valeurs
df.show(5, truncate=False)

only showing top 5 rows


In [10]:
# Preparation des donnees pour le partitionnement
# Conversion de tpep_pickup_datetime en timestamp
# Extraction du mois comme cle de partition
from pyspark.sql.functions import col, to_timestamp, month

df_casted = df.withColumn("Pickup_Datetime", 
    to_timestamp(col("tpep_pickup_datetime"), "MM/dd/yyyy hh:mm:ss a"))

df_final = df_casted.withColumn("Month", month(col("Pickup_Datetime"))) \
                    .filter(col("Month").isNotNull() & col("VendorID").isNotNull())

print("Colonnes Pickup_Datetime et Month creees")
df_final.select("VendorID", "tpep_pickup_datetime", "Month").show(5)

Colonnes Pickup_Datetime et Month creees


In [12]:
# Sauvegarde Parquet plat (sans partitionnement)
# Sert de reference pour le benchmark
df_final.write.mode("overwrite").parquet("hdfs://namenode:9000/data/taxi_flat.parquet")
print("Parquet plat sauvegarde sur HDFS")

Parquet plat sauvegarde sur HDFS


In [16]:
# Verification de la structure dans HDFS (a executer depuis le terminal)
!docker exec -it namenode hdfs dfs -ls /data/taxi_flat.parquet

/bin/sh: 1: docker: not found


In [18]:
# Sauvegarde Parquet partitionne : VendorID/Month = 24 partitions
df_final.write.mode("overwrite") \
    .partitionBy("VendorID", "Month") \
    .parquet("hdfs://namenode:9000/data/taxi_partitioned.parquet")
print("Parquet partitionne cree")

Parquet partitionne cree


In [21]:
# Benchmark 1 : CSV brut — lit tout le fichier (455 MB), pas de Partition Pruning
from pyspark.sql.functions import col, month, to_timestamp
import time

print(" Benchmark 1: CSV Brut ")

df_csv = spark.read.option("header", "true") \
    .csv("hdfs://namenode:9000/2023_Yellow_Taxi_Trip_Data.csv")

df_csv = df_csv.withColumn("Pickup_Timestamp", 
    to_timestamp(col("tpep_pickup_datetime"), "MM/dd/yyyy hh:mm:ss a"))

df_csv = df_csv.withColumn("Month", month(col("Pickup_Timestamp")))

start = time.time()
count_csv = df_csv.filter((col("VendorID") == "1") & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_csv}")
print(f"Temps: {end - start:.4f} secondes")

 Benchmark 1: CSV Brut 
Lignes trouvees: 827367
Temps: 2.9592 secondes


In [23]:
# Benchmark 2 : Parquet plat — compresse mais lit tous les fichiers (121 MB)
print("Benchmark 2: Parquet Plat ")

df_flat = spark.read.parquet("hdfs://namenode:9000/data/taxi_flat.parquet")

start = time.time()
count_flat = df_flat.filter((col("VendorID") == 1) & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_flat}")
print(f"Temps: {end - start:.4f} secondes")

Benchmark 2: Parquet Plat 
Lignes trouvees: 827367
Temps: 0.4295 secondes


In [25]:
# Benchmark 3 : Parquet partitionne — Partition Pruning actif, lit seulement 23.9 MB
print(" Benchmark 3: Parquet Partitionne ")

df_part = spark.read.parquet("hdfs://namenode:9000/data/taxi_partitioned.parquet")

start = time.time()
count_part = df_part.filter((col("VendorID") == 1) & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_part}")
print(f"Temps: {end - start:.4f} secondes")

 Benchmark 3: Parquet Partitionne 
Lignes trouvees: 827367
Temps: 0.2863 secondes
